[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q4/00_orient_and_precommit.ipynb)

# R1-Q4 Week 1 — Orient and precommit

### This notebook produces `precommit.json`, which holds the three decisions that drive every notebook after it.

The file this notebook produces will be read by every notebook that follows, so the decisions you make here are the rails the rest of the analysis runs on.

By the end of this notebook you will have:

- `precommit.json` containing your three locked-in decisions — data source plus stress-class scope, Wang 2023 168 h timepoint handling, and the "above-chance" null — plus the reasoning behind each, ready as Week 1 EQ#1 input.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Orient to the question (read the page, restate it in your own words)

Open the R1-Q4 page in Notion (the question this notebook is built around) and read it end to end before you write anything below. While you read, hold onto four things:

- **The question itself** — what's being asked, in one sentence.
- **The Background** — why this question is hard, and why *Arabidopsis* cold stress is a reasonable place to ask it. The CBF/DREB1 regulon is the foundational claim that makes cross-platform transfer plausible at all (Kidokoro et al., 2022).
- **The Prediction** — what you expect to see if transfer works, what "weak transfer" would look like, and what "no transfer" would look like.
- **The Considerations** — six bullets. Three become your precommit decisions later in this notebook (Sections 4, 5, and 6: data source plus stress-class scope, 168 h handling, and the "above-chance" null). The other three are directives, not choices: VST before ComBat, filter the AtGenExpress training set to shoot samples only, and note the ecotype provenance without overstating it.

In the cell below, write a short paragraph (3–5 sentences) restating the question in your own words. A good restatement names: (a) what the model is trained on and what it's tested on, (b) what "recognizing stress" means concretely — predicting a stress class from expression data, not a continuous score — and (c) the one outcome that would actually be informative: that the classifier identifies cold in Wang 2023 above the null you precommit to in Section 6.

This paragraph is for you. It isn't saved to `precommit.json`. Its job is to make sure you can argue the question is worth asking before you spend three weeks building the answer.

*(Write your restatement here. Replace this italicized text with your paragraph.)*

### 2) Inspect AtGenExpress experimental design (eight abiotic stresses × shoot/root × time-course, GEO range)

Before you run the cell below, open the **Rationale 1 datasets** page in Notion and read the AtGenExpress section. It covers the platform (ATH1 microarray, ~22k probes), the design (eight abiotic stresses × shoot/root × time-course), and the two download paths (GEO vs TAIR/NASCArrays) that matter for Precommit 1 in Section 4.

The next cell loads the AtGenExpress sample metadata and prints summary counts — the same information in tabular form. Run it once you've read the dataset page.

In [ ]:
# Load AtGenExpress sample metadata and print:
#   - total sample count
#   - samples per stress condition
#   - samples per tissue (shoot vs root)
#   - samples per timepoint
#   - whether oxidative stress is present
#     (its presence distinguishes the TAIR/NASCArrays 9-class scope
#      from the GEO 8-class scope — this is what Precommit 1 resolves)

Three things to hold onto from the output; they all feed into decisions later in this notebook or the next:

- **Stress conditions present.** AtGenExpress covers eight or nine abiotic stresses depending on download path. Nine if oxidative is included (TAIR/NASCArrays), eight if not (GEO). This is what Precommit 1 (Section 4) resolves.
- **Shoot vs root counts.** Wang 2023 sequenced rosette tissue, which is shoot-equivalent. NB01 Section 2 filters AtGenExpress to shoot samples only — this is a directive from the Considerations, not a choice. For now you just need to know the shoot/root split exists.
- **Timepoint range.** AtGenExpress timepoints top out at 24 h. Wang 2023 includes a 168 h timepoint that sits well outside this range. Precommit 2 (Section 5) decides how to handle it.

### 3) Inspect Wang 2023 experimental design (cold; rosette/shoot tissue; RNA-seq counts; 0 h, 2 h, 24 h, 168 h timepoints; ecotype provenance note — Col-0 confirmed indirectly via Guo et al. 2023, not stated in the data descriptor)

Open the **Rationale 1 datasets** page in Notion again, this time for the Wang 2023 section. It covers the platform (Illumina RNA-seq, count-scale data, ~33k genes), the design (cold vs control × four timepoints in rosette tissue), and one ecotype subtlety worth knowing before you proceed.

The next cell loads the Wang 2023 sample metadata and prints summary counts. Run it once you've read the dataset page.

Open the **Rationale 1 datasets** page in Notion again, this time for the Wang 2023 section. It covers the platform (Illumina RNA-seq, count-scale data, ~33k genes), the design (cold vs control × four timepoints in rosette tissue), and one ecotype subtlety worth knowing before you proceed.

The next cell loads the Wang 2023 sample metadata and prints summary counts. Run it once you've read the dataset page.

In [ ]:
# Load Wang 2023 sample metadata and print:
#   - total sample count
#   - samples per condition (cold vs control)
#   - samples per timepoint (0 h, 2 h, 24 h, 168 h)
#   - tissue source (expected: rosette)
#   - count-scale check (raw counts, not normalized intensities)

Five things to hold onto. Each one feeds into a decision later in this notebook or in NB01:

- **RNA-seq counts, not intensities.** Wang 2023 is Illumina RNA-seq on the count scale (~33k genes); AtGenExpress is ATH1 microarray on a log-intensity scale (~22k probes). This is why NB01 Section 4 applies VST to Wang 2023 first — count-scale variance has to be stabilized before ComBat can compare the two platforms.
- **Cold and control on the Wang side, eight or nine stress conditions on the AtGenExpress side.** Wang 2023 contains cold and control samples; the 0 h timepoint is the no-treatment baseline (Wang et al. 2023 Methods explicitly treats 0 h as "the control" in the DEG analysis). The SRA `isolate` field reads "cold stress 0h-1" for these samples because the whole experiment was deposited as a single cold-stress study, but the paper's analytical framing is unambiguous. Wang's 3 control × 9 cold support is class-imbalanced. This narrow-test-broad-train asymmetry, with imbalance on top, is why Precommit 3 (Section 6) has to define the "above-chance" null carefully.
- **Rosette tissue.** Rosette is shoot-equivalent. This is what makes the "filter AtGenExpress to shoot samples only" directive in NB01 Section 2 a real constraint — train and test need to share tissue, or any accuracy drop you see could be a tissue mismatch rather than a transfer failure.
- **Four timepoints, one of them awkward.** 0 h, 2 h, 24 h, 168 h. The 24 h timepoint is the headline — it overlaps the AtGenExpress range and is where the Prediction expects the strongest signal. The 168 h timepoint sits a week outside the AtGenExpress range, and Precommit 2 (Section 5) resolves how to handle it.
- **Ecotype provenance.** The Wang 2023 data descriptor does not state the ecotype. Col-0 is confirmed indirectly through Guo et al. (2023), a paper from the same research group. This is unlikely to affect transfer materially — both studies are almost certainly Col-0 — but you need to mention it in your methods so a reader can see what's confirmed directly versus inferred. Don't overstate.

### 4) Precommit 1: data source plus stress-class scope (GEO 8-class vs TAIR/NASCArrays 9-class)

This is Precommit 1, the first of three decisions you record before you run anything in NB01.

The AtGenExpress data lives in two places, and they don't contain the same set of stress conditions:

- **GEO range** — eight abiotic stresses (the eight you saw in Section 2's output, which does *not* include oxidative).
- **TAIR/NASCArrays** — all nine stresses, adding oxidative.

Either is defensible. The question is what scope claim you want to make in your paper. The decision matters less for what your classifier *can* predict (cold stress in Wang 2023, regardless of which source you choose) than for what it *cannot* predict (oxidative stress, if you go with GEO). Pick the download path that matches the claim you intend to make, and write down the reasoning before you run anything.

NB01 Section 2 reads this decision to choose which AtGenExpress download to load. (See Considerations bullet 6 on the R1-Q4 page for the long version.)

*(Write 2–3 sentences here naming your choice and the reasoning behind it. The code cell below records the same decision in machine-readable form — keep them consistent.)*

In [ ]:
# Initialize the precommit dict. This is the first precommit section,
# so the dict gets created here; Sections 5 and 6 will append to it,
# and Section 7 writes it to precommit.json.
#
# Record this section's decision under the "data_source_and_scope" key:
#   - source: "GEO" or "TAIR_NASCArrays"
#   - n_stress_classes: 8 (GEO) or 9 (TAIR_NASCArrays)
#   - rationale: 1-2 sentences in the student's own words,
#     consistent with the reflection cell above

### 5) Precommit 2: Wang 2023 168 h timepoint handling (exclude vs out-of-distribution probe)

This is Precommit 2, the second of three decisions you record.

Wang 2023 has four timepoints: 0 h, 2 h, 24 h, and 168 h. The first three sit inside AtGenExpress's range (which tops out at 24 h). The 168 h timepoint — a full week of cold — does not. Your classifier will never have seen anything that far into the time course, so what it does with those samples is genuinely uncertain.

Two defensible options:

- **Exclude.** Drop 168 h samples from the Wang 2023 test set entirely. Your transfer claim is then about 0 h, 2 h, and 24 h only.
- **Include as an out-of-distribution probe.** Keep 168 h in the test set. Treat it as a genuine extrapolation — the classifier should still call these samples "cold," but with degraded confidence, and any drop in accuracy at 168 h is a *finding* rather than a failure.

Pick one before you run anything. If you decide after seeing which option produces cleaner results, you've fit the method to the conclusion and your transfer claim stops being a real test. (See Considerations bullet 3 on the R1-Q4 page for the long version.)

NB01 Section 3 reads this decision to filter Wang 2023's timepoints on load. NB03 Section 2 reads it again when it breaks out per-timepoint accuracy.

*(Write 2–3 sentences here naming your choice and the reasoning behind it. The code cell below records the same decision in machine-readable form — keep them consistent.)*

In [ ]:
# Append this section's decision to the precommit dict that Section 4
# initialized. Do not re-initialize the dict here.
#
# Record under the "wang_168h_handling" key:
#   - decision: "exclude" or "include_as_ood"
#   - rationale: 1-2 sentences in the student's own words,
#     consistent with the reflection cell above

### 6) Precommit 3: "above-chance" null definition (binary cold-vs-control with a 50% null vs multi-class classifier behavior)

This is Precommit 3, the third and final decision you record.

You're training a multi-class classifier (8 or 9 stress conditions plus control, depending on Precommit 1) and testing it on Wang 2023, which contains cold and control samples. The training and test label spaces don't match: AtGenExpress is broad, Wang is narrow. That mismatch forces a choice about what "above chance" even means here.

The Wang test side is also class-imbalanced: 3 control samples (the 0 h timepoint) versus 9 cold samples (the 2 h, 24 h, and 168 h timepoints). Raw accuracy on this test set is misleading. A classifier that always predicts "cold_stress" — having learned nothing about cold biology — would score 9/12 ≈ 75 % raw accuracy. Without a metric that's insensitive to this imbalance, an "above 50 %" claim doesn't measure what it sounds like it measures.

Two defensible options:

- **Binary cold-vs-control balanced accuracy.** Collapse the classifier's multi-class output into a yes/no decision ("did it predict any stress, or did it predict control?"), then measure **balanced accuracy** — the mean of sensitivity (recall on cold) and specificity (recall on control). Balanced accuracy is insensitive to class imbalance by construction: the always-predict-cold classifier described above scores 50 % balanced accuracy, exactly the null. Null is **50 %**. Easy to summarize, easy to compare, easy to explain.
- **Multi-class per-class recall.** Keep the classifier's full multi-class output and report recall separately on the two Wang labels present: "what fraction of control samples got called 'control'?" and "what fraction of cold samples got called 'cold_stress'?" Each compared to a **1/N chance null** (1/8 or 1/9 depending on Precommit 1). This answers a richer question — does the classifier recognize cold stress *specifically*, or does it spread its predictions across the other stress conditions? — at the cost of being a pair of numbers rather than one.

Either is defensible. The first answers "does the classifier recognize *stress*?" The second answers "does the classifier recognize *cold stress* in particular?" — closer to the foundational claim about CBF/DREB1 that motivated the whole question, but harder to put in a headline figure. Pick one before you run anything, and write down both the null and the metric so the "above-chance" claim has a fixed referent the rest of the project can be measured against. (See Considerations bullet 4 on the R1-Q4 page for the long version.)

*(Write 2–3 sentences here naming your choice, the null you're committing to, and the reasoning behind it. The code cell below records the same decision in machine-readable form — keep them consistent.)*

In [ ]:
# Append this section's decision to the precommit dict that Section 4
# initialized. Do not re-initialize the dict here.
#
# Record under the "above_chance_null" key:
#   - decision: "binary_cold_vs_control" or "multiclass_behavior"
#   - null_description: string describing what the null is
#       e.g., "50% accuracy on a binary cold-vs-control test"
#       or "~1/N cold-call rate at uniform chance, where N is the
#       number of classes the classifier was trained on"
#   - metric: string describing what gets reported in Week 4
#       e.g., "binary accuracy"
#       or "prediction distribution across stress classes"
#   - rationale: 1-2 sentences in the student's own words,
#     consistent with the reflection cell above

### 7) Closeout: write `precommit.json`, submit EQ#1

You've reached the end of NB00. Three precommit decisions are now sitting in memory (Sections 4, 5, and 6 each appended to a running dict). What's left is to write them to disk, verify the file looks right, and submit EQ#1.

The cell below writes `precommit.json` to your R1-Q4 output directory (`OUTPUT_DIR`, set up in Cell 2). Run it, then open the file and read it back. Not because the write might fail silently — because EQ#1 is your record of what you committed to *before* you ran any analysis. NB01, NB02, and NB03 all open this file in their first body section. If a field is wrong here, every downstream notebook reads the wrong field.

In [ ]:
# Write the precommit dict to OUTPUT_DIR / "precommit.json".
# Use json.dump with indent=2 for human readability.
#
# Then read the file back from disk and pretty-print it,
# so you can visually confirm the three decisions and their
# rationales before submitting.
#
# Expected top-level keys (set in Sections 4, 5, 6):
#   - data_source_and_scope
#   - wang_168h_handling
#   - above_chance_null

Your EQ#1 deliverable has two parts: the `precommit.json` file you just wrote, and the three short reflection paragraphs you wrote in the markdown cells of Sections 4, 5, and 6 (those live inside the notebook itself). Submit per your program's EQ#1 procedure — see the iRI Mentor 2026 onboarding SOP if you're unsure where to upload.

Once EQ#1 is submitted, NB01 picks up next week. Its first body section is a defensive load of this same `precommit.json` — every decision you locked in here drives a branch downstream.